## 📦 Kütüphaneleri İçe Aktar

In [1]:
import numpy as np
import pandas as pd
import pickle
import warnings

from sklearn.pipeline import Pipeline

warnings.filterwarnings('ignore')

print("Libraries imported successfully!")

Libraries imported successfully!


## 📂 Model ve Preprocessor'ı Yükle

In [2]:
# Load original dataset to test raw input
DATA_PATH = '/kaggle/input/bank-customer-churn-dataset/Bank Customer Churn Prediction.csv'
df = pd.read_csv(DATA_PATH)

print(f"Dataset loaded: {df.shape}")
df.head()

Dataset loaded: (10000, 12)


,customer_id,credit_score,country,gender,age,tenure,balance,products_number,credit_card,active_member,estimated_salary,churn
0,15634602,619,France,Female,42,2,0.00,1,1,1,101348.88,1
1,15647311,608,Spain,Female,41,1,83807.86,1,0,1,112542.58,0
2,15619304,502,France,Female,42,8,159660.80,3,1,0,113931.57,1
3,15701354,699,France,Female,39,1,0.00,2,0,0,93826.63,0
4,15737888,850,Spain,Female,43,2,125510.82,1,1,1,79084.10,0


## 🔧 Özellik Mühendisliği Fonksiyonu Tanımla

In [3]:
def apply_feature_engineering(df):
    """
    Apply the same feature engineering from notebook 3
    """
    df_fe = df.copy()
    
    # 1. Balance to Salary Ratio
    df_fe['balance_to_salary_ratio'] = df_fe['balance'] / (df_fe['estimated_salary'] + 1)
    
    # 2. Tenure to Age Ratio
    df_fe['tenure_age_ratio'] = df_fe['tenure'] / (df_fe['age'] + 1)
    
    # 3. Credit Score Category
    df_fe['credit_score_category'] = pd.cut(
        df_fe['credit_score'], 
        bins=[0, 600, 700, 850], 
        labels=['Low', 'Medium', 'High']
    )
    
    # 4. Age Group
    df_fe['age_group'] = pd.cut(
        df_fe['age'], 
        bins=[0, 30, 50, 100], 
        labels=['Young', 'Middle-aged', 'Senior']
    )
    
    # 5. High Value Customer
    balance_threshold = df_fe['balance'].quantile(0.75)
    salary_threshold = df_fe['estimated_salary'].quantile(0.75)
    df_fe['high_value_customer'] = (
        (df_fe['balance'] >= balance_threshold) & 
        (df_fe['estimated_salary'] >= salary_threshold)
    ).astype(int)
    
    # 6. Inactive High Balance
    df_fe['inactive_high_balance'] = (
        (df_fe['active_member'] == 0) & 
        (df_fe['balance'] >= balance_threshold)
    ).astype(int)
    
    return df_fe

print("Feature engineering function defined!")

Feature engineering function defined!


## 🔧 Özellik Kolon Listelerini Tanımla

In [4]:
# NOT: Notebook 3 ve 4'ü "Add Data" ile ekleyin!

# Notebook 3'ten preprocessor.pkl
preprocessor_path = '/kaggle/input/3-feature-engineering-notebook/preprocessor.pkl'

# Notebook 4'ten best_model.pkl
model_path = '/kaggle/input/4-model-optimization-notebook/best_model.pkl'

# Load preprocessor
with open(preprocessor_path, 'rb') as f:
    preprocessor = pickle.load(f)

# Load best model
with open(model_path, 'rb') as f:
    best_model = pickle.load(f)

print("✅ Preprocessor and model loaded successfully!")
print(f"\nPreprocessor: {type(preprocessor).__name__}")
print(f"Model: {type(best_model).__name__}")

✅ Preprocessor and model loaded successfully!

Preprocessor: ColumnTransformer
Model: CatBoostClassifier


## 🏗️ ChurnPredictionPipeline Sınıfı Oluştur

In [5]:
class ChurnPredictionPipeline:
    """
    Complete pipeline: raw input → feature engineering → preprocessing → model → prediction
    """
    
    def __init__(self, preprocessor, model):
        self.preprocessor = preprocessor
        self.model = model
        
        # Define feature columns
        self.original_numerical = ['credit_score', 'age', 'tenure', 'balance', 'products_number', 'estimated_salary']
        self.original_categorical = ['country', 'gender']
        self.original_binary = ['credit_card', 'active_member']
        
        self.new_numerical = ['balance_to_salary_ratio', 'tenure_age_ratio']
        self.new_categorical = ['credit_score_category', 'age_group']
        self.new_binary = ['high_value_customer', 'inactive_high_balance']
        
        self.all_numerical = self.original_numerical + self.new_numerical
        self.all_categorical = self.original_categorical + self.new_categorical
        self.all_binary = self.original_binary + self.new_binary
        
        self.all_features = self.all_numerical + self.all_categorical + self.all_binary
    
    def apply_feature_engineering(self, df):
        """
        Apply feature engineering
        """
        df_fe = df.copy()
        
        # 1. Balance to Salary Ratio
        df_fe['balance_to_salary_ratio'] = df_fe['balance'] / (df_fe['estimated_salary'] + 1)
        
        # 2. Tenure to Age Ratio
        df_fe['tenure_age_ratio'] = df_fe['tenure'] / (df_fe['age'] + 1)
        
        # 3. Credit Score Category
        df_fe['credit_score_category'] = pd.cut(
            df_fe['credit_score'], 
            bins=[0, 600, 700, 850], 
            labels=['Low', 'Medium', 'High']
        )
        
        # 4. Age Group
        df_fe['age_group'] = pd.cut(
            df_fe['age'], 
            bins=[0, 30, 50, 100], 
            labels=['Young', 'Middle-aged', 'Senior']
        )
        
        # 5. High Value Customer
        balance_threshold = df_fe['balance'].quantile(0.75)
        salary_threshold = df_fe['estimated_salary'].quantile(0.75)
        df_fe['high_value_customer'] = (
            (df_fe['balance'] >= balance_threshold) & 
            (df_fe['estimated_salary'] >= salary_threshold)
        ).astype(int)
        
        # 6. Inactive High Balance
        df_fe['inactive_high_balance'] = (
            (df_fe['active_member'] == 0) & 
            (df_fe['balance'] >= balance_threshold)
        ).astype(int)
        
        return df_fe
    
    def predict(self, X):
        """
        Make predictions from raw input
        """
        # Convert to DataFrame if needed
        if isinstance(X, dict):
            X = pd.DataFrame([X])
        elif not isinstance(X, pd.DataFrame):
            X = pd.DataFrame(X)
        
        # Apply feature engineering
        X_fe = self.apply_feature_engineering(X)
        
        # Select features
        X_selected = X_fe[self.all_features]
        
        # Preprocess
        X_preprocessed = self.preprocessor.transform(X_selected)
        
        # Predict
        predictions = self.model.predict(X_preprocessed)
        
        return predictions
    
    def predict_proba(self, X):
        """
        Get prediction probabilities from raw input
        """
        # Convert to DataFrame if needed
        if isinstance(X, dict):
            X = pd.DataFrame([X])
        elif not isinstance(X, pd.DataFrame):
            X = pd.DataFrame(X)
        
        # Apply feature engineering
        X_fe = self.apply_feature_engineering(X)
        
        # Select features
        X_selected = X_fe[self.all_features]
        
        # Preprocess
        X_preprocessed = self.preprocessor.transform(X_selected)
        
        # Predict probabilities
        probabilities = self.model.predict_proba(X_preprocessed)
        
        return probabilities

print("Custom pipeline class defined!")

Custom pipeline class defined!


## 🚀 Pipeline Örneği Oluştur

In [6]:
# Instantiate final pipeline
final_pipeline = ChurnPredictionPipeline(preprocessor, best_model)

print("✅ Final pipeline created!")
print(f"\nPipeline components:")
print(f"  1. Feature Engineering (built-in)")
print(f"  2. Preprocessor: {type(preprocessor).__name__}")
print(f"  3. Model: {type(best_model).__name__}")

✅ Final pipeline created!

Pipeline components:
  1. Feature Engineering (built-in)
  2. Preprocessor: ColumnTransformer
  3. Model: CatBoostClassifier


## 🧪 Örnek Girdi ile Pipeline'ı Test Et

In [7]:
# Test with a single sample (raw input)
test_sample = df.iloc[0:1].drop('churn', axis=1)

print("Testing pipeline with raw input...\n")
print("Input:")
print(test_sample)

# Predict
prediction = final_pipeline.predict(test_sample)
probability = final_pipeline.predict_proba(test_sample)

print(f"\n✅ Prediction successful!")
print(f"\nPrediction: {prediction[0]} ({'Churned' if prediction[0] == 1 else 'Not Churned'})")
print(f"Probability [Not Churn, Churn]: {probability[0]}")
print(f"Churn Probability: {probability[0][1]:.4f}")

Testing pipeline with raw input...

Input:
   customer_id  credit_score country  gender  age  tenure  balance  \
0     15634602           619  France  Female   42       2      0.0   

   products_number  credit_card  active_member  estimated_salary  
0                1            1              1         101348.88  

✅ Prediction successful!

Prediction: 0 (Not Churned)
Probability [Not Churn, Churn]: [0.66448099 0.33551901]
Churn Probability: 0.3355


In [8]:
# Test with dictionary input
test_dict = {
    'credit_score': 650,
    'country': 'France',
    'gender': 'Male',
    'age': 35,
    'tenure': 5,
    'balance': 100000,
    'products_number': 2,
    'credit_card': 1,
    'active_member': 1,
    'estimated_salary': 75000
}

print("\nTesting pipeline with dictionary input...\n")
print("Input:")
print(test_dict)

# Predict
prediction = final_pipeline.predict(test_dict)
probability = final_pipeline.predict_proba(test_dict)

print(f"\n✅ Prediction successful!")
print(f"\nPrediction: {prediction[0]} ({'Churned' if prediction[0] == 1 else 'Not Churned'})")
print(f"Probability [Not Churn, Churn]: {probability[0]}")
print(f"Churn Probability: {probability[0][1]:.4f}")


Testing pipeline with dictionary input...

Input:
{'credit_score': 650, 'country': 'France', 'gender': 'Male', 'age': 35, 'tenure': 5, 'balance': 100000, 'products_number': 2, 'credit_card': 1, 'active_member': 1, 'estimated_salary': 75000}

✅ Prediction successful!

Prediction: 0 (Not Churned)
Probability [Not Churn, Churn]: [0.95926875 0.04073125]
Churn Probability: 0.0407


## 💾 Final Pipeline'ı Kaydet

In [9]:
# Save final pipeline
output_dir = '/kaggle/working/'

with open(output_dir + 'final_pipeline.pkl', 'wb') as f:
    pickle.dump(final_pipeline, f)

print("✅ Final pipeline saved successfully!")
print(f"\nFile: final_pipeline.pkl")
print(f"\nThis pipeline can be used for deployment:")
print(f"  - Load: pickle.load(open('final_pipeline.pkl', 'rb'))")
print(f"  - Predict: pipeline.predict(raw_input)")
print(f"  - Probabilities: pipeline.predict_proba(raw_input)")

✅ Final pipeline saved successfully!

File: final_pipeline.pkl

This pipeline can be used for deployment:
  - Load: pickle.load(open('final_pipeline.pkl', 'rb'))
  - Predict: pipeline.predict(raw_input)
  - Probabilities: pipeline.predict_proba(raw_input)


## 📝 Final Pipeline Özeti

### Pipeline Bileşenleri:

#### 1. Özellik Mühendisliği
Pipeline şunları otomatik olarak oluşturur:
- **balance_to_salary_ratio**: Bakiye-maaş oranı
- **tenure_age_ratio**: Müşteri olma süresi-yaş oranı
- **credit_score_category**: Kredi skoru kategorileri
- **age_group**: Yaş grupları
- **high_value_customer**: Yüksek değerli müşteri flag'i
- **inactive_high_balance**: Aktif olmayan + yüksek bakiye flag'i

#### 2. Ön İşleme
- **Sayısal Özellikler**: RobustScaler ile ölçeklendirme
- **Kategorik Özellikler**: OneHotEncoding (drop='first')
- **Binary Özellikler**: Olduğu gibi

#### 3. Tahmin Modeli
- En iyi performans gösteren model (XGBoost/LightGBM/CatBoost)
- Optimize edilmiş hiperparametreler

### Kullanım:

```python
# Pipeline'ı yükle
pipeline = load_pipeline()

# Tek tahmin
input_data = {
    'credit_score': 650,
    'age': 35,
    ...
}
prediction = pipeline.predict(input_data)
probability = pipeline.predict_proba(input_data)

# Batch tahmin
df = pd.DataFrame([...])
predictions = pipeline.predict(df)
```

### Kaydedilen Dosyalar:
✅ `final_pipeline.pkl` - Tam end-to-end pipeline

### Deployment'a Hazır:
Bu pipeline artık şunlar için kullanıma hazır:
- 🌐 **Streamlit Web Uygulaması** (app.py)
- 📊 **Batch Tahmin** (inference.py)
- 🔌 **API Entegrasyonu**
- 📦 **Production Deployment**

### Sonraki Adımlar:
1. **Scripts klasörünü kullan** - config.py, pipeline.py, inference.py, app.py
2. **Streamlit uygulamasını çalıştır** - `streamlit run app.py`
3. **Production'a deploy et** - Docker, cloud platformları

🎉 Pipeline hazır ve deployment için tamamen paketlenmiş!